In [27]:
import trafilatura
import logging
import boto3
from ingestion.producer.s3client import S3Config
from typing import Any
import json
from psycopg2.extras import execute_batch
import os
import requests

In [ ]:
SAMPLE_DATA = "test_news_data.json"
import dotenv
dotenv.load_dotenv("./.env", override=True)
# print(os.environ)

In [4]:
def read_sample_data(file_path: str = SAMPLE_DATA) -> list[dict]:
    with open(file_path, "r") as f:
        data = json.load(f)
    return data

In [36]:

def download_content(url: str, fallback_content:str ="Fetch failed from url") -> str | None:

    headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}
    if not url:
        return fallback_content
    try:
        response = requests.get(
        url,
        headers=headers,
        timeout=15,)
        html = response.text
        if response.status_code != 200:
            logging.warning(f"Failed to fetch {url}, status code: {response.status_code}")
            return fallback_content
        if html:
            return trafilatura.extract(html, include_tables=True, output_format='html')
    except Exception as e:
        logging.warning(f"Error fetching content from {url}: {e}")
    return fallback_content

In [35]:
download_content("https://finnhub.io/api/news?id=cce2b8118f4ec0562be2b34fb75a4187fb2c2fd4ce6fa84035917edcd60d451e")

403


'<html>\n  <body>\n    <p>Enable JavaScript and cookies to continue</p>\n  </body>\n</html>'

In [6]:
def get_s3_client():
    config = S3Config.from_env()
    s3_kwargs: dict[str, Any] = {
        "region_name": config.region,
        "endpoint_url": config.endpoint_url,
        "aws_access_key_id": config.access_key_id,
        "aws_secret_access_key": config.secret_access_key,
        "verify": config.verify_ssl,
    }
    s3_kwargs = {k: v for k, v in s3_kwargs.items() if v is not None}
    return boto3.client("s3", **s3_kwargs), config.bucket


In [7]:
import psycopg2


def get_pg_connection():
    return psycopg2.connect(
        host=os.getenv("PG_HOST", "192.168.2.100"),
        port=int(os.getenv("PG_PORT", 5432)),
        dbname=os.getenv("PG_DB", "news_db"),
        user=os.getenv("PG_USER", "postgres"),
        password=os.getenv("PG_PASSWORD", "postgres"),
    )

In [38]:
def process_s3_key_to_postgres(s3_key: str) -> int:
    """Đọc file trực tiếp từ S3, cào trafilatura và nạp vào Postgres."""
    s3_client, bucket_name = get_s3_client()

    logging.info(f"Đang tải {s3_key} từ bucket {bucket_name}...")
    obj = s3_client.get_object(Bucket=bucket_name, Key=s3_key)
    raw_content = obj["Body"].read().decode("utf-8")
    data = json.loads(raw_content)

    items = data if isinstance(data, list) else [data]
    if not items:
        return 0

    records = []
    for item in items:
        # Cào full content từ SequentNews.ipynb
        url = item.get("url")
        content = download_content(url, fallback_content=item.get("content", ""))

        records.append({
            "id": str(item.get("id")),
            "source": item.get("raw_payload", {}).get("source"),
            "title": item.get("title"),
            "content": content,
            "url": url,
            "published_at": item.get("published_at"),
            "image_url": item.get("image_url"),
            "category": item.get("category"),
            "symbol": item.get("symbol", ""),
            "ingested_at": item.get("ingested_at"),
            "raw_payload": json.dumps(item.get("raw_payload", {})),
        })

    upsert_sql = """
    INSERT INTO news_articles (
        id, source, title, content, url,
        published_at, image_url, category, symbol,
        ingested_at
    ) VALUES (
        %(id)s, %(source)s, %(title)s, %(content)s, %(url)s,
        %(published_at)s, %(image_url)s, %(category)s, %(symbol)s,
        %(ingested_at)s
    )
    ON CONFLICT (id) DO UPDATE SET
        content = EXCLUDED.content,
        title = EXCLUDED.title,
        image_url = EXCLUDED.image_url,
        ingested_at = EXCLUDED.ingested_at;
    """

    conn = get_pg_connection()
    with conn.cursor() as cur:
        execute_batch(cur, upsert_sql, records, page_size=100)
    conn.commit()
    conn.close()

    logging.info(f"Đã lưu thành công {len(records)} records từ {s3_key} vào Postgres.")
    return len(records)

In [39]:
process_s3_key_to_postgres("news_raw/src=finnhub/AAPL/2026-10-05.json")

43